# SteamRec – Notebook voor content-based & ML.NET collaborative modellen

Deze notebook behandelt zowel het **content-based model (SteamRec.Core)** als het **ML.NET collaborative filtering model (SteamRec.ML)** en volgt de volledige workflow:
- Data-analyse
- Selectie, exploratie en preprocessing
- Opzetten van de ML pipeline (train/test splits, cross-validation, evaluatie)
- Hyperparameter-optimalisatie
- Evaluatie van resultaten (Precision/Recall, MAE, RMSE)
- Ondersteuning voor forecasting-gebaseerde aanbevelingen
- Visualisatie


## 1) Data laden uit MongoDB
Gebruik onderstaande code om `games` en `interactions` in te laden.


In [ ]:
import os
import pandas as pd
from pymongo import MongoClient

MONGO_URI = os.getenv("MONGODB_CONNECTION_STRING", "mongodb://localhost:27017")
DB_NAME = "steamrec-test"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

games = pd.DataFrame(list(db["games"].find({})))
interactions = pd.DataFrame(list(db["interactions"].find({})))

games.head()


## 2) Data-analyse & exploratie
We starten met een globale analyse: aantallen, missende waarden, basisstatistieken en verdelingen.


In [ ]:
games.info()
games.describe(include="all").T


In [ ]:
interactions.info()
interactions.describe(include="all").T


### Voorbeeld-visualisaties
De volgende grafieken helpen om scheefheid in playtime te zien en om populaire genres/tags te identificeren.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(interactions.get("playtime_forever", pd.Series(dtype=float)).fillna(0), bins=50, log_scale=(False, True))
plt.title("Verdeling playtime_forever (log y)")
plt.show()


## 3) Selectie, cleaning & preprocessing (Garbage in – Garbage out)
We filteren ongeldige records, normaliseren velden en maken features.


In [ ]:
# Basis cleaning
games_clean = games.dropna(subset=["app_id"]).copy()
interactions_clean = interactions.dropna(subset=["steam_id", "app_id"]).copy()

# Converteer IDs naar consistente types
games_clean["app_id"] = games_clean["app_id"].astype(int)
interactions_clean["app_id"] = interactions_clean["app_id"].astype(int)
interactions_clean["steam_id"] = interactions_clean["steam_id"].astype(str)

# Verwijder zero-interacties (geen signaal)
if "playtime_forever" in interactions_clean.columns:
    interactions_clean = interactions_clean[interactions_clean["playtime_forever"] > 0]

games_clean.head()


## 4) Content-based model (SteamRec.Core)
Het content-based model zit in **`SteamRec.Core`**. Hieronder staat een C#-voorbeeld voor het bouwen van een content-based recommender.

> **Tip:** compileer eerst het project in `csharp/SteamRec.Core`.

```csharp
using SteamRec.Core;

// Map MongoDB data naar GameRecord lijst
var games = gamesClean.Select(row => new GameRecord
{
    AppId = row.AppId,
    Name = row.Name,
    Genres = row.Genres,
    Tags = row.Tags,
    Categories = row.Categories,
    ReviewScoreAdj = row.ReviewScoreAdj,
    ReviewVolumeLog = row.ReviewVolumeLog
}).ToList();

var recommender = new ContentBasedRecommender(games);
var similar = recommender.RecommendSimilar(appId: 570, topN: 10);
var personal = recommender.RecommendForLiked(new[] { 570, 730 }, topN: 20);
```

### Evaluatie (precision/recall) voor content-based
We kunnen content-based aanbevelingen evalueren met held-out interacties (bijv. 80/20 split per user).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

# Voorbeeld: train/test split per user (hier simplistisch)
user_ids = interactions_clean["steam_id"].unique().tolist()
train_users, test_users = train_test_split(user_ids, test_size=0.2, random_state=42)

train_interactions = interactions_clean[interactions_clean["steam_id"].isin(train_users)]
test_interactions = interactions_clean[interactions_clean["steam_id"].isin(test_users)]

# Placeholder: y_true/y_pred bereken je op basis van aanbevelingen vs. echte aankopen/plays
y_true = []
y_pred = []

# precision = precision_score(y_true, y_pred)
# recall = recall_score(y_true, y_pred)
# print(precision, recall)


## 5) ML.NET Collaborative Filtering (SteamRec.ML)
Het collaborative model is geïmplementeerd in **`SteamRec.ML`** met ML.NET matrix factorization.

```csharp
using SteamRec.ML;

var cf = new CollaborativeFilteringRecommender();

// rows: (steamId, appId, playtimeForever, playtime2Weeks)
cf.TrainFromRows(rows);

// evaluatie: train/test split + scoring
var recommendations = cf.RecommendForUser("7656119...", candidateAppIds, excludeAppIds, topN: 20);
```

### Train/test split + cross-validation
Voor implicit feedback kun je per gebruiker een **leave-one-out** of **80/20 split** doen en de rest van de interacties gebruiken om te trainen.


In [ ]:
# Voorbeeld: train/test split op interacties
train_interactions, test_interactions = train_test_split(interactions_clean, test_size=0.2, random_state=42)

# Cross-validation kan via K-fold per user (pseudocode)
# for fold in folds:
#   train on fold, test on fold


### Evaluatiemetrics (MAE, RMSE, Precision/Recall)
Bij collaborative filtering gebruik je vaak **RMSE** op ratings/scores en **Precision/Recall@K** voor top-K aanbevelingen.


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# y_true = echte interacties/ratings
# y_pred = voorspelde scores
y_true = np.array([1, 0, 1, 1])
y_pred = np.array([0.9, 0.1, 0.8, 0.4])

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

print("MAE", mae)
print("RMSE", rmse)


## 6) Hyperparameter-optimalisatie
Voor ML.NET kunnen we een eenvoudige grid-search doen op **rank**, **alpha** en **lambda**.

```csharp
var options = new MatrixFactorizationTrainer.Options
{
    NumberOfIterations = 30,
    ApproximationRank = rank,
    Alpha = alpha,
    Lambda = lambda
};

// Loop over combinaties en log RMSE/Precision@K
```

Voor content-based kunnen we de gewichten (similarity, review score, review volume) tunen op basis van validatie-precision.


## 7) Forecasting-gebaseerde aanbevelingen (seizoensgebonden trends)
Wanneer `interactions` tijdstempels bevat, kunnen we trends detecteren.

Voorbeeld: seizoensgebonden playtime per genre en toekomstig momentum detecteren.


In [ ]:
# Verondersteld: interactions_clean bevat een timestamp-kolom "timestamp"
if "timestamp" in interactions_clean.columns:
    interactions_clean["timestamp"] = pd.to_datetime(interactions_clean["timestamp"])
    interactions_clean["month"] = interactions_clean["timestamp"].dt.to_period("M").dt.to_timestamp()

    monthly = (interactions_clean
               .groupby("month")["playtime_forever"]
               .sum()
               .reset_index())

    plt.figure(figsize=(9, 4))
    sns.lineplot(data=monthly, x="month", y="playtime_forever")
    plt.title("Maandelijkse totale playtime (trend)")
    plt.show()


## 8) Visualisatie van aanbevelingen
Gebruik grafieken om de top-N aanbevelingen per model te vergelijken.


In [ ]:
# Voorbeelddata
topn_content = pd.DataFrame({"app_id": [1,2,3], "score": [0.95, 0.90, 0.88]})
topn_cf = pd.DataFrame({"app_id": [2,4,5], "score": [0.92, 0.89, 0.85]})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=topn_content, x="app_id", y="score", ax=axes[0])
axes[0].set_title("Content-based top-N")

sns.barplot(data=topn_cf, x="app_id", y="score", ax=axes[1])
axes[1].set_title("Collaborative top-N")

plt.tight_layout()
plt.show()


## 9) Samenvatting
Deze notebook laat de volledige workflow zien en gebruikt **SteamRec.Core** en **SteamRec.ML** als basis voor respectievelijk content-based en collaborative aanbevelingen.
Pas de evaluatie aan op jullie dataset (bv. expliciete ratings vs. implicit playtime).
